# MilliMetre: Results Walkthrough

This is a companion to the main risk-aware last-mile routing notebook. The main notebook generates the simulated delivery world, trains the models, plans routes, and saves evaluation files. This notebook reads those saved files and explains the results in smaller steps.

**Data honesty:** road geometry comes from OpenStreetMap. Customers, parcels, delivery outcomes, traffic, rider GPS traversals, and operating costs are simulated. These results describe this simulator; they are not proof of savings in a real courier operation.

## What we will inspect

1. The generated data and delivery failure patterns.
2. The failure-risk model and road travel-time model metrics.
3. The five route planners and paired test-day comparisons.
4. The stress test, business sensitivity, and one sample rider route.

## 0. Run the main notebook first

Open the original notebook, lastmile_pipeline.ipynb, and run it from top to bottom. It creates a working folder containing:

- data/ — observable simulated input tables;
- outputs/ — model metrics, planner evaluations, and sample rider plans;
- artifacts/ — saved model weights and configuration.

The main notebook uses lastmile_run/ locally and /content/lastmile/ in Google Colab. This notebook searches for either location. It does not recreate the data or retrain the models.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import stats

plt.rcParams.update({
    "figure.figsize": (10, 4),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Set this manually if the generated run folder is somewhere else.
PROJECT_DIR = None

if PROJECT_DIR is None:
    candidates = []
    env_path = os.environ.get("MILLIMETRE_RUN_DIR")
    if env_path:
        candidates.append(Path(env_path).expanduser())
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend([base, base / "lastmile_run", base / "lastmile"])
    candidates.extend([Path("/content/lastmile"), Path("/content/lastmile_run")])
    PROJECT_DIR = next(
        (p.resolve() for p in candidates
         if (p / "data" / "delivery_attempts.csv").exists()
         and (p / "outputs" / "planner_results_by_day.csv").exists()),
        None,
    )

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Run lastmile_pipeline.ipynb first, or set PROJECT_DIR to the folder "
        "containing data/ and outputs/."
    )

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
print("Using generated run:", PROJECT_DIR)

## 1. Inspect the generated data

The road network is real map geometry. The operational records are generated by a simulator. We load only observable tables from data/; the hidden ground-truth files are not used in this analysis notebook.

In [ ]:
required_files = [
    "road_nodes.csv", "road_edges.csv", "weather.csv", "customers.csv",
    "riders.csv", "manifests.csv", "delivery_attempts.csv", "gps_traversals.csv",
]
missing = [name for name in required_files if not (DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing generated data files: {missing}")

nodes = pd.read_csv(DATA_DIR / "road_nodes.csv")
edges = pd.read_csv(DATA_DIR / "road_edges.csv")
weather = pd.read_csv(DATA_DIR / "weather.csv")
customers = pd.read_csv(DATA_DIR / "customers.csv")
riders = pd.read_csv(DATA_DIR / "riders.csv")
manifests = pd.read_csv(DATA_DIR / "manifests.csv")
attempts = pd.read_csv(DATA_DIR / "delivery_attempts.csv")
gps = pd.read_csv(DATA_DIR / "gps_traversals.csv")

inventory = pd.DataFrame({
    "table": ["road nodes", "road edges", "customers", "riders", "weather days",
              "parcel manifests", "delivery attempts", "GPS traversals"],
    "rows": [len(nodes), len(edges), len(customers), len(riders), len(weather),
             len(manifests), len(attempts), len(gps)],
})
display(inventory)
print(f"Road graph: {nodes.node_id.nunique():,} nodes and {len(edges):,} directed edges")
print(f"Simulation period: {weather.date.min()} to {weather.date.max()}")

## 2. Delivery outcomes: what fails?

A delivery attempt is failed when its outcome is marked failed. For first-attempt analysis, we keep only attempt_no equal to 1. Re-attempts should not be mixed with the original delivery decision.

In [ ]:
attempts["failed"] = (attempts["outcome"] == "failed").astype(int)
first_attempts = attempts.loc[attempts["attempt_no"] == 1].copy()

summary = pd.DataFrame({
    "measure": ["All attempts", "First attempts", "Unique customers",
                "Unique riders", "First-attempt failure rate"],
    "value": [len(attempts), len(first_attempts), customers.customer_id.nunique(),
              riders.rider_id.nunique(), first_attempts.failed.mean()],
})
display(summary)

reason_counts = (
    first_attempts.loc[first_attempts.failed == 1, "failure_reason"]
    .fillna("not recorded")
    .value_counts()
    .rename_axis("failure reason")
    .to_frame("failed first attempts")
)
display(reason_counts)

## 3. Does arrival time matter?

The failure model estimates a failure probability at different possible arrival times. A basic diagnostic is to compare first-attempt failure rates by arrival hour. This chart is descriptive; it does not isolate the effect of arrival time from customer type, parcel type, or day.

In [ ]:
hourly_failure = (
    first_attempts.assign(hour=first_attempts.arrival_hour.astype(int))
    .groupby("hour", as_index=False)
    .agg(failure_rate=("failed", "mean"), attempts=("failed", "size"))
)
display(hourly_failure)

fig, ax = plt.subplots()
ax.plot(hourly_failure.hour, hourly_failure.failure_rate, marker="o")
ax.set(title="First-attempt failure rate by arrival hour",
       xlabel="Arrival hour", ylabel="Share of first attempts that failed")
ax.set_ylim(bottom=0)
ax.grid(alpha=0.25)
plt.show()

## 4. Failure reasons and payment type

The simulator records not_home, refused, and address_issue as failure reasons. The next table compares the outcome mix between COD and prepaid deliveries.

In [ ]:
payment = first_attempts["is_cod"].map({0: "Prepaid", 1: "COD"})
payment = payment.fillna(first_attempts["is_cod"].astype(str))
reason_mix = pd.crosstab(
    payment,
    first_attempts["failure_reason"].fillna("delivered"),
    normalize="index",
)
reason_order = [c for c in ["delivered", "not_home", "refused", "address_issue"]
                if c in reason_mix.columns]
display(reason_mix[reason_order].round(3))

reason_mix[reason_order].plot(kind="bar", stacked=True, figsize=(9, 4))
plt.title("First-attempt outcome mix by payment type")
plt.xlabel("Payment type")
plt.ylabel("Share of first attempts")
plt.legend(title="Outcome", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 5. Failure-risk model performance

The main notebook compares logistic regression, XGBoost, and a GRU-based model called FailNet. It also includes a base-rate model that assigns the same probability to every attempt.

- PR-AUC should be compared with the test failure rate.
- ROC-AUC of 0.5 is random ranking.
- Log-loss and Brier score evaluate the quality of predicted probabilities.
- Capture top 10% is the fraction of all test failures found among the 10% of attempts ranked as riskiest.

In [ ]:
failure_path = OUTPUT_DIR / "failure_model_metrics.csv"
if failure_path.exists():
    failure_metrics = pd.read_csv(failure_path, index_col=0)
    display(failure_metrics.sort_values("test_PR_AUC", ascending=False).round(4))

    plot_cols = [c for c in ["test_PR_AUC", "test_ROC_AUC"]
                 if c in failure_metrics.columns]
    if plot_cols:
        failure_metrics[plot_cols].plot(kind="bar", figsize=(9, 4))
        plt.title("Failure-risk model metrics on held-out test attempts")
        plt.ylabel("Score")
        plt.xticks(rotation=20, ha="right")
        plt.grid(axis="y", alpha=0.25)
        plt.tight_layout()
        plt.show()
else:
    print("failure_model_metrics.csv not found. Run the main notebook through evaluation.")

## 6. Road travel-time model: does the GNN generalise?

Intersections are graph nodes and road segments are directed edges. The main notebook compares free-flow estimates, a road-class lookup table, an MLP without message passing, and a GNN.

The held-out-edge metric is especially useful: it measures performance on roads with no training GPS traversals. In the saved run, the GNN performs better on seen edges, but the MLP has lower error on held-out edges. The GNN is therefore not best on every generalisation check.

In [ ]:
travel_path = OUTPUT_DIR / "travel_time_metrics.csv"
if travel_path.exists():
    travel_metrics = pd.read_csv(travel_path, index_col=0)
    display(travel_metrics.round(4))

    metric = "MAE log-ratio (held-out edges)"
    if metric in travel_metrics.columns:
        travel_metrics[metric].sort_values().plot(kind="bar", figsize=(8, 4))
        plt.title("Travel-time error on held-out road edges")
        plt.ylabel("Mean absolute log-ratio error (lower is better)")
        plt.xticks(rotation=20, ha="right")
        plt.grid(axis="y", alpha=0.25)
        plt.tight_layout()
        plt.show()
else:
    print("travel_time_metrics.csv not found.")

## 7. Compare the five route planners

- P1 — Distance-only VRP: reference planner using free-flow travel estimates.
- P2 — Time-aware (GNN): adds time-dependent road travel predictions.
- P3a — Risk-aware sequencing: adds the learned delivery-failure probability.
- P3b — Risk-aware with deferral: can also postpone a parcel to tomorrow.
- P4 — Oracle risk: uses the simulator's true failure probabilities. This is an upper-bound diagnostic, not a deployable model.

Every planner receives the same held-out parcel list and is evaluated under the same simulated customer and traffic draws.

In [ ]:
summary_path = OUTPUT_DIR / "planner_summary.csv"
results_path = OUTPUT_DIR / "planner_results_by_day.csv"
if not summary_path.exists() or not results_path.exists():
    raise FileNotFoundError("Planner results are missing. Run the main notebook through evaluation.")

planner_summary = pd.read_csv(summary_path, index_col=0)
planner_results = pd.read_csv(results_path)

show_cols = [c for c in [
    "cost / day (₹)", "cost / parcel (₹)", "vs P1 (₹/day)", "95% CI ±",
    "vs P2 (₹/day)", "95% CI ± (P2)", "first-attempt fail rate", "overtime min"
] if c in planner_summary.columns]
display(planner_summary[show_cols].round(2))

## 8. The key comparison: P1 → P3b and P2 → P3b

These are different questions:

1. **P1 → P3b:** does the complete system improve on distance-only routing?
2. **P2 → P3b:** what extra value does the learned failure-risk layer add after time-aware road travel is already included?

We calculate paired savings by test day. Positive values mean the candidate costs less. The confidence interval is based on the ten paired daily differences in the saved run.

In [ ]:
base_results = planner_results.loc[planner_results.scenario == "base"].copy()

def paired_saving(frame, baseline, candidate):
    wide = frame.loc[
        frame.planner.isin([baseline, candidate]),
        ["day", "planner", "cost_total"],
    ].pivot(index="day", columns="planner", values="cost_total").dropna()

    saving = wide[baseline] - wide[candidate]
    n = len(saving)
    mean = saving.mean()
    margin = (stats.t.ppf(0.975, n - 1) * saving.std(ddof=1) / np.sqrt(n)
              if n > 1 else np.nan)
    return {
        "comparison": f"{baseline} → {candidate}",
        "test days": n,
        "mean saving per day (₹)": mean,
        "95% CI lower (₹)": mean - margin,
        "95% CI upper (₹)": mean + margin,
    }

P1 = "P1 Distance-only VRP"
P2 = "P2 Time-aware (GNN)"
P3 = "P3b Risk-aware (+deferral)"
comparisons = pd.DataFrame([
    paired_saving(base_results, P1, P3),
    paired_saving(base_results, P2, P3),
])
display(comparisons.round(2))

### What the saved run actually says

In the saved ten-day evaluation, average simulated cost is about ₹10,451 per hub-day for P1, ₹9,582 for P2, and ₹9,552 for P3b.

The full system comparison (P1 → P3b) saves about ₹899 per hub-day. The incremental comparison (P2 → P3b) saves only about ₹29 per hub-day, and its 95% confidence interval includes zero.

**A fair interpretation:** time-aware routing accounts for most of the measured improvement in this simulation. The additional benefit of the learned failure-risk layer is not established by this ten-day test.

In [ ]:
planner_summary["cost / day (₹)"].sort_values().plot(kind="barh", figsize=(9, 4))
plt.title("Mean simulated operating cost per hub-day")
plt.xlabel("Cost (₹); lower is better")
plt.ylabel("")
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## 9. Behaviour-shift stress test

The main notebook also re-scores plans in a deliberately misspecified world: customers return later, some leave earlier, irregular customers behave less predictably, and COD refusals become more common.

This is a stress test under the simulator's chosen perturbations. It is not a guarantee of robustness to every kind of real-world distribution shift.

In [ ]:
misspecified = planner_results.loc[
    planner_results.scenario == "misspecified"
].copy()

if not misspecified.empty:
    stress = (
        misspecified.groupby("planner", as_index=False)
        .agg(
            mean_cost_per_day=("cost_total", "mean"),
            failure_rate=("fail_rate", "mean"),
            deferred_per_day=("deferred", "mean"),
        )
        .sort_values("mean_cost_per_day")
    )
    display(stress.round(2))
else:
    print("No misspecified scenario rows found.")

## 10. Business sensitivity is not a revenue forecast

The original notebook re-prices the same simulated outcomes under different assumed costs per failed attempt. This shows how the financial interpretation changes when the penalty for a failure changes. It does not establish what a courier will pay or validate those assumptions.

In [ ]:
business_path = OUTPUT_DIR / "business_numbers.csv"
if business_path.exists():
    business = pd.read_csv(business_path)
    display(business.round(2))
else:
    print("business_numbers.csv not found.")

## 11. Inspect an exported rider plan

The main pipeline exports rider ID, stop order, planned arrival time, predicted risk, and customer coordinates. We can inspect an example and optionally draw the route on a map.

The route line is a visual connection between stops. It is not a street-level navigation engine.

In [ ]:
plan_files = sorted(OUTPUT_DIR.glob("plan_*.csv"))
if plan_files:
    plan_path = plan_files[0]
    plan = pd.read_csv(plan_path)
    print("Example plan:", plan_path.name)
    print("Planned stops:", len(plan))
    display(plan.head(12))

    hub_file = DATA_DIR / "hub.csv"
    if hub_file.exists():
        try:
            import folium
            hub = pd.read_csv(hub_file).iloc[0]
            route_map = folium.Map(
                location=[hub.lat, hub.lon], zoom_start=14,
                tiles="cartodbpositron"
            )
            palette = ["blue", "green", "purple", "darkred", "orange", "cadetblue"]
            for i, (rider, group) in enumerate(plan.groupby("rider_id")):
                points = [[hub.lat, hub.lon]] + group[["lat", "lon"]].values.tolist()
                color = palette[i % len(palette)]
                folium.PolyLine(points, color=color, weight=3,
                                opacity=0.7, tooltip=str(rider)).add_to(route_map)
                for _, stop in group.iterrows():
                    stop_color = "red" if stop.risk_at_arrival > 0.30 else color
                    folium.CircleMarker(
                        [stop.lat, stop.lon], radius=4, color=stop_color,
                        fill=True, fill_opacity=0.8,
                        tooltip=(f"{rider}, stop {int(stop.seq)}, "
                                 f"{stop.planned_arrival}, risk {stop.risk_at_arrival:.2f}"),
                    ).add_to(route_map)
            folium.Marker([hub.lat, hub.lon], tooltip="Delivery hub").add_to(route_map)
            display(route_map)
        except ImportError:
            print("Install folium to view the map: pip install folium")
else:
    print("No plan_*.csv found. Run the route-export section in the main notebook.")

## 12. What this experiment does not prove

1. **Synthetic operations:** customers, delivery attempts, traffic, and GPS traces were generated by a simulator. The real-world input is road geometry from OpenStreetMap.
2. **Short evaluation window:** the planner comparison uses ten held-out test days in the saved run. A larger evaluation is needed.
3. **The GNN is not uniformly best:** it performs better on seen-edge error but worse on the held-out-edge metric in the saved run.
4. **The risk layer has a small measured increment:** P2 → P3b has a confidence interval that includes zero in this run.
5. **Costs are assumptions:** rider-minute costs, fuel costs, failure penalties, and service assumptions must be calibrated with real operators.

## A sensible real-world pilot

Obtain consented, anonymised hub data. Compare the current planning process with this system on alternating days or riders. Keep the same service area and parcel definitions. Measure cost per delivered parcel, first-attempt success, distance, rider-hours, overtime, and deferred parcels. Predefine the evaluation period and report uncertainty, not only a point estimate.

**Bottom line:** this is a useful research prototype for combining time-dependent routing with delivery-risk prediction. The next step is to test its assumptions and collect real operational evidence.